Retrieval-Augmented Generation (RAG)

Document loading

In [1]:

from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"  

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- page preview Pg-2 ---")
print(pages[1].page_content[:10000])

C:\Users\Pinakesh\AppData\Local\Temp\ipykernel_29952\3271948378.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 9 pages from the PDF.

--- page preview Pg-2 ---
Telecom Technical Reference Guide  - Internal Use Only
1. Introduction to Mobile Networks
Mobile networks have evolved through several generations, each offering significant improvements in speed,
capacity, and capability.
2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to
around 50 kbps, sufficient only for text messaging and simple email.
3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, and app
downloads at speeds of 1-14 Mbps. This generation established the foundation for smartphone adoption
worldwide.
4G (LTE) networks delivered a major leap forward, with typical download speeds of 20-150 Mbps and latency
below 50 ms. LTE networks are fully IP-based, meaning voice calls are transmitted as data (VoLTE) rather than
over a separate circuit-switched channel. LTE-Advanced (LTE-A) uses carrier aggregation to combine multiple
fr

Chunking

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,      
    chunk_overlap=100,   
    separators=["\n\n", "\n", ".", " "],  # tries paragraph → line → sentence → word
)

chunks = splitter.split_documents(pages)

len(chunks)


37

In [3]:
print("____________________________________________________________")
print()
print(chunks[0].page_content)
print()
print("____________________________________________________________")
print()
print(chunks[1].page_content)
print()
print("____________________________________________________________")
print()
print(chunks[2].page_content)
print()
print("____________________________________________________________")
print()
print(chunks[3].page_content)
print()

____________________________________________________________

Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1

____________________________________________________________

Telecom Technical Reference Guide  - Internal Use Only
1. Introduction to Mobile Networks
Mobile networks have evolved through several generations, each offering significant improvements in speed,
capacity, and capability.
2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to
around 50 kbps, sufficient only for text messaging and simple email.
3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, and app

____________________________________________________________

downloads at speeds of 1-14 Mbps. This generation established the foundation for smartphone adoption
worldwide.
4G (LTE) n

Creating Vector Embeddings for the Chunks

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

c:\Users\Pinakesh\Desktop\GenAI\Calling_LLM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5443.03it/s]


Vector store ready. 37 vectors stored.


Retriving Relative Chunks according to Query

In [5]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

--- Chunk 3 ---
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call cannot be established  - for exam



In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

True

In [7]:
# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# --- LLM via Groq API ---
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    reasoning_format="parsed",
)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

RAG chain assembled.


In [9]:
questions = input("Enter your Question")

print(f"Q: {questions}\nA: {chain.invoke(questions)}\n")

Q: what is this application about
A: Based on the information in the reference guide, the “MyTelecom app” is the telecom provider’s customer‑facing mobile application that lets subscribers log in to view and manage their account. Through the app customers can:

* Access their plan details and make changes (e.g., upgrades or downgrades)  
* Update personal information such as address or SIM‑card settings  
* Review usage and billing information  

Because the app provides direct control over the account, the guide highlights security concerns (e.g., account‑takeover attacks) and recommends safeguards such as mandatory two‑factor authentication, anomaly detection for logins from new devices or locations, and step‑up authentication for high‑risk actions.



Done